# Engram: can Gemma fill the Knowledge State?

Gemma reads each chat in conversations 0 to 4 and writes structured observations, which code checks. No answers are written. Benchmark data only.

**Kaggle:** Settings > Accelerator > GPU T4 x2, and Settings > Turn on internet. Then Run > Run all. Keep the tab open.

In [ ]:
COMMIT = "8b10edb"
import os, subprocess, time, urllib.request
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu: raise SystemExit("No GPU. Choose Settings > Accelerator > GPU T4 x2.")
print("GPU:", gpu.replace(chr(10), ", "))
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception:
    raise SystemExit("No internet. Choose Settings > Turn on internet, then run again.")
WORK = "/kaggle/working"
os.chdir(WORK)
subprocess.run("rm -rf Engram; git clone -q https://github.com/manjunathhari/Engram.git", shell=True, check=True)
os.chdir(f"{WORK}/Engram")
subprocess.run(f"git checkout -q {COMMIT}; git clone -q --depth 1 https://github.com/snap-research/locomo.git", shell=True, check=True)
subprocess.run("pip install -q wordllama scikit-learn spacy; python -m spacy download en_core_web_sm", shell=True, check=True, capture_output=True)
import wordllama, shutil
p = os.path.dirname(wordllama.__file__)
a, b = os.path.join(p, "tokenizers"), os.path.join(p, "tokenizer")
if os.path.isdir(a) and not os.path.exists(b): shutil.copytree(a, b)
subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{WORK}/ollama.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(5)
MODEL = "hf.co/bartowski/google_gemma-3-4b-it-GGUF:Q4_K_M"
r = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
if r.returncode: raise SystemExit("Model download failed: " + r.stderr[-200:])
subprocess.run(["ollama", "cp", MODEL, "gemma3:4b"])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
print("setup ready")

In [ ]:
%%writefile /kaggle/working/knowledge_test.py
"""Can Gemma fill the Knowledge State? It reads each chat in conversations 0-4 and writes structured observations, which learn() checks.
Measures what is accepted and rejected, for a blind check of correctness afterwards. Benchmark data only. No answers are written."""
import argparse, json, os, re, sys, tempfile, time
from collections import Counter
sys.path.insert(0, os.getcwd())
import vps_kit as K
import engram_chat as E
import engram_knowledge as S
from engram import Config

ap = argparse.ArgumentParser()
ap.add_argument("--url", required=True); ap.add_argument("--model", required=True); ap.add_argument("--save", required=True)
a = ap.parse_args()
ids = [0, 1, 2, 3, 4]
cfg = Config(**json.load(open("final_config.json")))
tmp = tempfile.mkdtemp()
convs = {c: K.load_conv(K.DATA[c]) for c in ids}
llm = K.LLM(a.url, a.model, "local", 1200)
PROMPT = """Below is one conversation. Each line starts with its id. Write the lasting knowledge in it as structured lines.

Use exactly this form, one per line, at most 12 lines:
ID | subject | word | object | time

ID: copy the id of the line that states it.
subject: a person's full name as written. Never he, she, they, I, or my friend; use the speaker's name for I.
word: choose only from this list:
  people and places: owns, lives_in, works_at, married_to, parent_of, friend_of, member_of
  details: job_title, status, likes, dislikes, hobby, plans, number
  events: adopted, bought, sold, lost, gave_away, moved_to, started_job, left_job, got_married, divorced, joined, left_group, started_hobby, stopped_hobby, attended, visited, completed
object: the name, place or short value, using the same words as the line.
time: the real date if the line states one or a date is in square brackets, otherwise write unknown.

Example:
D3:7 | Anna Lee | adopted | Rex | 7 May 2023
D3:9 | Anna Lee | hobby | pottery | unknown

Write only what is stated. Do not guess. If nothing fits the list, leave it out. Write nothing else.

Conversation:
{chat}

Lines:
"""
LINE_ID = re.compile(r"\bD\d+:\d+\b")
cache_path = f"{a.save}/knowledge_replies.json"
cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}


def parse(line):
    parts = [p.strip().strip('"*') for p in line.strip().strip("*-• ").split("|")]
    if len(parts) < 4: return None
    sources = LINE_ID.findall(parts[0])
    if not sources: return None
    word = re.sub(r"[\s\-]+", "_", parts[2].lower()).strip("_")
    time_text = parts[4] if len(parts) > 4 else ""
    time_text = "" if time_text.lower() in ("unknown", "none", "n/a", "") else time_text
    o = dict(subject=parts[1], sources=sources, time=time_text)
    if word in S.EVENTS: o.update(event=word, object=parts[3])
    elif word in S.RELATIONS: o.update(predicate=word, object=parts[3])
    else: o.update(predicate=word, value=parts[3])
    return o


results = []
reasons = Counter()
def dump():
    json.dump(dict(chats=len(cache), lines=len(results), accepted=sum(r["accepted"] for r in results), reasons=reasons.most_common(), seconds=round(sum(v["secs"] for v in cache.values()) / max(1, len(cache)), 1), results=results), open(f"{a.save}/knowledge_results.json", "w"))


for c in ids:
    m = E.ChatMemory(f"{tmp}/{c}.db", cfg=cfg, durable=False)
    ks = S.KnowledgeState(m.db)
    by = {}
    for t in convs[c].turns: by.setdefault(t["session"], []).append(t)
    for ses, ts in by.items():
        d = ts[0]["date"]
        m.open_chat(d)
        for t in ts: m.add(t["speaker"], t["text"], date=d, ref=t["id"], caption=t["cap"], commit=False)
        m.db.commit(); m.close_chat(write_note=False, sleep=False)
        ref = dict(m.db.execute("SELECT ref, sid FROM sentences").fetchall())
        key = f"{c}:{ses}"
        if key not in cache:
            chat = f"[{d}]\n" + "\n".join(f"{t['id']} {t['speaker']}: {E.with_dates(t['text'], d)}" + (f" [image: {t['cap']}]" if t["cap"] else "") for t in ts)
            s0 = time.time()
            r = llm(PROMPT.format(chat=chat), max_tokens=700)[0]
            cache[key] = dict(raw=r, secs=round(time.time() - s0, 1))
            json.dump(cache, open(cache_path, "w"))
        for line in cache[key]["raw"].splitlines():
            if not line.strip(): continue
            o = parse(line)
            if o is None:
                results.append(dict(chat=key, line=line[:300], accepted=False, reason="line not in the form asked")); reasons["line not in the form asked"] += 1
                continue
            refs = o["sources"]
            o["sources"] = [ref[x] for x in refs if x in ref and x.startswith(f"D{ses}:")]
            out = ks.learn(o)
            why = re.sub(r":.*", "", out.get("reason", ""))
            if not out["accepted"]: reasons[why] += 1
            results.append(dict(chat=key, line=line[:300], refs=refs, accepted=out["accepted"], reason=out.get("reason"), time_status=out.get("time_status"), note=out.get("note"), kind=out.get("kind")))
        print(f"chat {key}: {len(results)} lines so far", flush=True)
        dump()
    states = {r[0]: ks.get_state(r[0]) for r in m.db.execute("SELECT name FROM entities").fetchall()}
    json.dump(states, open(f"{a.save}/knowledge_state_{c}.json", "w"))
print("Done", flush=True)

In [ ]:
import os, subprocess, sys, time
WORK, SAVE = "/kaggle/working", "/kaggle/working/engram_results"
os.makedirs(SAVE, exist_ok=True); os.chdir(f"{WORK}/Engram")
LOG = f"{SAVE}/knowledge_run.log"
if subprocess.run(["pgrep", "-f", "knowledge_test.py"], capture_output=True).returncode != 0:
    subprocess.Popen([sys.executable, "-u", f"{WORK}/knowledge_test.py", "--url", "http://localhost:11434/v1", "--model", "gemma3:4b", "--save", SAVE], stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True)
    print("test started")
for _ in range(960):
    time.sleep(15)
    t = open(LOG).read().splitlines()
    print(chr(13) + " | ".join(t[-1:])[:150], end="")
    if t and (t[-1].startswith("Done") or "STOPPED" in t[-1] or "Traceback" in "\n".join(t[-30:])): break
print(); print("\n".join(open(LOG).read().splitlines()[-25:])); print(sorted(os.listdir(SAVE)))

When it says Done, send Claude `final_answers.jsonl`, `facts_final.json` and `knowledge_run.log` from the `engram_results` folder in the Output panel.